# Transfer Learning Image Classifier — Complete Training Pipeline

This notebook walks through the entire ML pipeline, step by step, with explanations of **why** each decision was made — not just what the code does.

**Pipeline overview:**
1. Configuration and reproducibility setup
2. Dataset download and exploration
3. Data augmentation preview
4. Phase 1 — Feature extraction (frozen backbone)
5. Phase 2 — Fine-tuning (unfrozen top layers)
6. Baseline CNN training (from scratch)
7. Model evaluation and error analysis
8. Grad-CAM interpretability
9. Results comparison and discussion

---
## 1. Configuration and Reproducibility

Every hyperparameter lives in `config.yaml` — not hard-coded in the training script. This makes experiments auditable and reproducible: when someone asks "what learning rate did you use?", the answer is always in one file.

We also set all random seeds upfront. This locks down randomness in Python, NumPy, PyTorch, and cuDNN so that the same code produces the same results across runs.

In [ ]:
# Optional: if you see ModuleNotFoundError (timm, matplotlib, etc.), uncomment and run once,
# then restart the kernel and run the notebook from the top.
# %pip install -q -r ../requirements.txt

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.utils import set_seed, load_config, get_device, ensure_dirs

config = load_config('../configs/config.yaml')
set_seed(config['project']['seed'])
device = get_device(config['project']['device'])
ensure_dirs(config)

print(f'Device : {device}')
print(f'Seed   : {config["project"]["seed"]}')
print(f'Mixed precision: {config["training"].get("mixed_precision", False)}')

---
## 2. Dataset Download and Exploration

We use the **Intel Image Classification** dataset — 6 natural scene categories with ~25,000 images. The dataset is downloaded automatically via `kagglehub`.

**Why re-split?** The original Kaggle dataset has only train and test folders, with no validation set. Tuning hyperparameters on the test set would leak information and inflate our reported metrics. We merge everything and re-split 70/15/15 with stratified per-class sampling.

In [ ]:
from src.data_loader import download_dataset, prepare_splits, get_dataloaders

source_path = download_dataset(config)
stats = prepare_splits(source_path, config['data']['data_dir'], config)

for split, counts in stats.items():
    total = sum(counts.values())
    print(f'{split:>5s}: {total:>5d} images  - {dict(counts)}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

train_counts = stats['train']
classes = sorted(train_counts.keys())
values = [train_counts[c] for c in classes]

fig, ax = plt.subplots(figsize=(8, 4))
colours = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4', '#FF5722']
bars = ax.bar(classes, values, color=colours)
ax.set_title('Training Set - Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Images')
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

---
## 3. Data Augmentation Preview

Augmentation artificially expands the training distribution by applying plausible real-world transformations. The model never sees the exact same image twice, which forces it to learn features that generalise rather than memorise.

| Augmentation | Purpose |
|---|---|
| RandomHorizontalFlip | Scenes are valid when mirrored |
| RandomRotation (±15°) | Simulates camera tilt |
| RandomResizedCrop | Different scales and partial views |
| ColorJitter | Different lighting and weather conditions |

Validation and test images are **not** augmented — evaluation must be deterministic.

In [ ]:
from src.augmentations import get_train_transforms, get_val_transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid
import torch

train_loader, val_loader, test_loader = get_dataloaders(config)

def show_batch(loader, title, n=8):
    images, labels = next(iter(loader))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    images = images[:n] * std + mean
    grid = make_grid(images, nrow=4).permute(1,2,0).clamp(0,1).numpy()
    plt.figure(figsize=(12,6))
    plt.imshow(grid)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()

show_batch(train_loader, 'Augmented Training Samples')

---
## 4. Phase 1 — Feature Extraction (Frozen Backbone)

The pretrained EfficientNetB0 backbone is **frozen** so that gradient updates only flow through the classifier head (~660 K parameters out of 4.7 M total).

**Why freeze?** The backbone already contains excellent feature extractors learned from 1.2 M ImageNet images. Training them with a high learning rate on our small dataset would overwrite those features — this is called **catastrophic forgetting**. Freezing prevents this entirely.

We use a learning rate of `1e-3` because the classifier head is randomly initialised and needs to converge quickly.

In [ ]:
from src.model import build_model
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from src.utils import EarlyStopping
from src.train import run_phase
from torch.utils.tensorboard import SummaryWriter

model = build_model(config).to(device)
criterion = nn.CrossEntropyLoss()

p1 = config['training']['phase1']
sched_cfg = config['training']['scheduler']
es_cfg = config['training']['early_stopping']

model.freeze_backbone()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.1f}%)')

use_amp = config['training'].get('mixed_precision', False) and device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda') if use_amp else None

opt1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=p1['learning_rate'], weight_decay=p1['weight_decay']
 )
sched1 = ReduceLROnPlateau(opt1, factor=sched_cfg['factor'], patience=sched_cfg['patience'])
es1 = EarlyStopping(patience=es_cfg['patience'], min_delta=es_cfg['min_delta'])

writer = SummaryWriter(log_dir=config['paths']['tensorboard'])

h1 = run_phase('phase1', model, train_loader, val_loader, criterion,
               opt1, sched1, device, config, writer, p1['epochs'], es1,
               scaler=scaler)

print(f"\nPhase 1 best val acc: {max(h1['val_acc']):.4f}")

---
## 5. Phase 2 — Fine-Tuning (Unfrozen Top Layers)

We now unfreeze the top 20 parameters of the backbone and train with a **100× smaller learning rate** (`1e-5`).

**Why so small?**
The unfrozen layers are already near a good optimum from ImageNet pretraining. Large gradient updates would push them away from that optimum — this is the fine-tuning form of catastrophic forgetting. A tiny LR ensures gradual, controlled adaptation.

**Why only the top layers?**
Lower layers encode universal features (edges, textures) that transfer well to any domain. Higher layers encode ImageNet-specific semantics that benefit from domain adaptation.

In [ ]:
p2 = config['training']['phase2']
model.unfreeze_backbone(from_layer=p2.get('unfreeze_from', -20))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable after unfreeze: {trainable:,} / {total:,} params')
opt2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=p2['learning_rate'], weight_decay=p2['weight_decay']
 )
sched2 = ReduceLROnPlateau(opt2, factor=sched_cfg['factor'], patience=sched_cfg['patience'])
es2 = EarlyStopping(patience=es_cfg['patience'], min_delta=es_cfg['min_delta'])
h2 = run_phase('phase2', model, train_loader, val_loader, criterion,
               opt2, sched2, device, config, writer, p2['epochs'], es2,
               scaler=scaler)
writer.close()
history = {k: h1[k] + h2[k] for k in h1}
print(f"\nFinal best val acc: {max(history['val_acc']):.4f}")

---
## 6. Baseline CNN (Trained from Scratch)

To quantify the benefit of transfer learning, we train a simple 3-block CNN on the same data with the same augmentations.

This model has no pretrained knowledge — it must learn every visual feature from our ~17 K training images alone. With only 3 convolutional layers, its receptive field and representational capacity are far smaller than EfficientNetB0.

In [ ]:
from src.baseline_model import BaselineCNN

baseline = BaselineCNN(num_classes=config['model']['num_classes']).to(device)
bcfg = config['baseline']

b_total = sum(p.numel() for p in baseline.parameters())
print(f'Baseline CNN: {b_total:,} params (all trainable)')

b_opt = torch.optim.Adam(baseline.parameters(), lr=bcfg['learning_rate'],
                         weight_decay=bcfg['weight_decay'])
b_sched = ReduceLROnPlateau(b_opt, factor=sched_cfg['factor'], patience=sched_cfg['patience'])
b_es = EarlyStopping(patience=es_cfg['patience'], min_delta=es_cfg['min_delta'])

b_writer = SummaryWriter(log_dir=os.path.join(config['paths']['tensorboard'], 'baseline'))

b_history = run_phase('baseline', baseline, train_loader, val_loader, criterion,
                      b_opt, b_sched, device, config, b_writer, bcfg['epochs'], b_es)
b_writer.close()

print(f"Baseline best val acc: {max(b_history['val_acc']):.4f}")

---
## 7. Evaluation and Error Analysis

A single accuracy number is not enough. We compute precision, recall, and F1 to understand per-class performance, and generate a confusion matrix to see exactly which classes are confused.

In [ ]:
from src.evaluate import evaluate_model, plot_confusion_matrix
from src.utils import load_checkpoint

class_names = config['model']['class_names']
models_dir = config['paths']['models']
plots_dir = config['paths']['plots']

# Load best checkpoints (saved during training) so we evaluate best models, not last epoch
best_phase2_path = os.path.join(models_dir, 'best_phase2.pth')
best_baseline_path = os.path.join(models_dir, 'best_baseline.pth')

if os.path.isfile(best_phase2_path):
    load_checkpoint(model, best_phase2_path, optimizer=None, device=device)
    print('Loaded best transfer-learning checkpoint.')
tl_results = evaluate_model(model, test_loader, device, class_names=class_names)
plot_confusion_matrix(tl_results['confusion_matrix'], class_names,
                     save_path=os.path.join(plots_dir, 'confusion_matrix_tl.png'))
print(f"Transfer Learning — Accuracy: {tl_results['accuracy']:.4f}, F1: {tl_results['f1']:.4f}\n")

if os.path.isfile(best_baseline_path):
    load_checkpoint(baseline, best_baseline_path, optimizer=None, device=device)
    print('Loaded best baseline checkpoint.')
bl_results = evaluate_model(baseline, test_loader, device, class_names=class_names)
plot_confusion_matrix(bl_results['confusion_matrix'], class_names,
                     save_path=os.path.join(plots_dir, 'confusion_matrix_baseline.png'))
print(f"Baseline CNN — Accuracy: {bl_results['accuracy']:.4f}, F1: {bl_results['f1']:.4f}")

---
## 8. Grad-CAM Interpretability

Grad-CAM overlays reveal which spatial regions drive the model's prediction. A well-trained model should attend to the **dominant subject** of the scene — trees for forests, waves for sea, buildings for buildings — not background noise.

If the heatmaps consistently highlight irrelevant regions (e.g. the sky for every class), that signals a data or model issue worth investigating.

In [ ]:
from src.gradcam import visualize_gradcam_batch

visualize_gradcam_batch(
    model, test_loader, device, class_names,
    num_images=6, save_dir='../outputs/gradcam'
)

from IPython.display import Image as IPImage, display
display(IPImage(filename='../outputs/gradcam/gradcam_results.png'))

---
## 9. Results Comparison and Discussion

In [ ]:
# Comparison table (requires tl_results and bl_results from Section 7 evaluation cell)
import pandas as pd

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Baseline CNN': [
        f"{bl_results['accuracy']:.4f}",
        f"{bl_results['precision']:.4f}",
        f"{bl_results['recall']:.4f}",
        f"{bl_results['f1']:.4f}",
    ],
    'Transfer Learning': [
        f"{tl_results['accuracy']:.4f}",
        f"{tl_results['precision']:.4f}",
        f"{tl_results['recall']:.4f}",
        f"{tl_results['f1']:.4f}",
    ],
})
print(comparison.to_string(index=False))

### Why Transfer Learning Outperforms the Baseline

1. **Pretrained feature hierarchy** — EfficientNetB0 has already learned edges, textures, shapes, and object parts from 1.2 M ImageNet images. The baseline must learn everything from ~17 K images.

2. **Data efficiency** — with a strong feature extractor in place, the classifier head needs far fewer examples to converge. The baseline overfits more easily because it has no prior knowledge.

3. **Implicit regularisation** — the frozen backbone in Phase 1 acts as a regulariser. Only a small number of parameters update, which limits the model's capacity to memorise training noise.

4. **Controlled fine-tuning** — Phase 2 uses a 100× smaller learning rate to gently adapt the backbone's higher-level features without destroying universal low-level features.

5. **Architectural depth** — EfficientNetB0 is far deeper than the 3-layer baseline, giving it a richer hierarchy of feature representations and a much larger receptive field.